In [1]:
%pip install -q langchain transformers langchain-huggingface langchain-community langchain-core langchain-text-splitters bitsandbytes docx2txt langchain-chroma

Note: you may need to restart the kernel to use updated packages.


In [2]:
from dotenv import load_dotenv

load_dotenv()

True

In [3]:
from langchain_huggingface import ChatHuggingFace, HuggingFacePipeline

llm = HuggingFacePipeline.from_model_id(
    model_id="microsoft/Phi-3-mini-4k-instruct",
    task="text-generation",
    device=0,
    pipeline_kwargs=dict(
        max_new_tokens=512,
        do_sample=False,
        repetition_penalty=1.03,
    ),
)


c:\Users\JJH\anaconda3\envs\langchain\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 195/195 [00:00<00:00, 4062.51it/s]
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Passing `generation_config` together with generation-related arguments=({'max_new_tokens', 'repetition_penalty', 'do_sample'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


In [4]:
chat_model = ChatHuggingFace(llm=llm)

In [5]:
ai_message = chat_model.invoke("what is huggingface?")

Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


In [6]:
ai_message.content

"<|user|>\nwhat is huggingface?<|end|>\n<|assistant|>\n Hugging Face is a company and community that provides AI models and tools for natural language processing (NLP) tasks. It was founded by Ilya Sutskever, Opus Tateishi, and Ilya Sutskever in 2018. The company's main product is the Transformers library, which includes pre-trained models like BERT, GPT-2, and RoBERTa that can be used for various NLP tasks such as text classification, question answering, and text generation.\n\nHugging Face also offers an open-source platform called the Hugging Face Hub, where users can share their own models and datasets with the community. This allows researchers, developers, and data scientists to collaborate on building and improving NLP models.\n\nIn addition to its products and services, Hugging Face has been actively involved in promoting ethical AI practices and ensuring that its models are accessible to everyone. They have also partnered with organizations like Microsoft and Google to integra

In [7]:
from transformers import BitsAndBytesConfig

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype="float16",
    bnb_4bit_use_double_quant=True,
)

In [8]:
quantized_llm = HuggingFacePipeline.from_model_id(
    model_id="microsoft/Phi-3-mini-4k-instruct",
    task="text-generation",
    device=0,
    pipeline_kwargs=dict(
        max_new_tokens=512,
        do_sample=False,
        repetition_penalty=1.03,
    ),
    
    model_kwargs={"quantization_config": quantization_config},
)


Exception in thread Thread-9 (_readerthread):
Traceback (most recent call last):
  File "c:\Users\JJH\anaconda3\envs\langchain\Lib\threading.py", line 1075, in _bootstrap_inner
    self.run()
  File "c:\Users\JJH\anaconda3\envs\langchain\Lib\threading.py", line 1012, in run
    self._target(*self._args, **self._kwargs)
  File "c:\Users\JJH\anaconda3\envs\langchain\Lib\subprocess.py", line 1599, in _readerthread
    buffer.append(fh.read())
                  ^^^^^^^^^
  File "<frozen codecs>", line 322, in decode
UnicodeDecodeError: 'utf-8' codec can't decode byte 0xc0 in position 6: invalid start byte
Loading weights: 100%|██████████| 195/195 [00:02<00:00, 82.77it/s]
Setting the `device` argument to None from 0 to avoid the error caused by attempting to move the model that was already loaded on the GPU using the Accelerate module to the same or another device.


In [9]:
quantized_chat_model = ChatHuggingFace(llm=quantized_llm)

In [10]:
quantized_ai_message = quantized_chat_model.invoke("what is huggingface")

Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
